# Dataset 4 — Cleaning Notebook

**Dataset:** Large non-experimental data (historic, retrospective use)
**Purpose:** Apply all cleaning steps identified during the audit.
**Inspection notebook:** `03_dataset_4_inspection.ipynb`
**Audit report:** `outputs/reports/ds4_audit_report.md`

Outputs written to `outputs/cleaned/`:
- `ds4_behavioral_clean.csv` + `ds4_behavioral_clean.md`
- `ds4_survey_clean.csv` + `ds4_survey_clean.md`
- `ds4_participant_linkage.csv` + `ds4_participant_linkage.md`

Raw files in `data_raw/` are never modified.

## 0. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT    = Path("..").resolve()
DS4_RAW = ROOT / "data_raw" / "4. data set, large non-experimental data"
CLEANED = ROOT / "outputs" / "cleaned"
CLEANED.mkdir(parents=True, exist_ok=True)

BEH_CSV    = DS4_RAW / "Behavior data set" / "behavior_data_set_4.csv"
SURVEY_CSV = DS4_RAW / "Survey data set"   / "post_survey_data_set_4.csv"

print("Project root:", ROOT)
print("Cleaned output folder:", CLEANED)

---
## 1. Load Raw Files

Delimiter note: behavioral is COMMA-delimited; survey is SEMICOLON-delimited.

In [ ]:
beh_raw    = pd.read_csv(BEH_CSV, sep=",", low_memory=False)
survey_raw = pd.read_csv(SURVEY_CSV, sep=";")

print(f"Behavioral raw:  {beh_raw.shape[0]:,} rows x {beh_raw.shape[1]} cols")
print(f"Survey raw:      {survey_raw.shape[0]} rows x {survey_raw.shape[1]} cols")

---
## 2. Behavioral Cleaning
### 2.1 Parse timestamp to UTC datetime

In [ ]:
beh = beh_raw.copy()

beh["timestamp_utc"] = pd.to_datetime(beh["timestamp"], utc=True)

print("Sample timestamps:")
print(beh[["timestamp", "timestamp_utc"]].head(3).to_string())
print()
print(f"Date range: {beh['timestamp_utc'].min()} to {beh['timestamp_utc'].max()}")
print(f"Parse failures: {beh['timestamp_utc'].isna().sum()}")

### 2.2 Fix app name typo

Create canonical `app_name` column correcting `whastapp` -> `whatsapp`.
Original `app` column is preserved.

In [ ]:
typo_map = {"whastapp": "whatsapp"}
beh["app_name"] = beh["app"].replace(typo_map)

print("Original 'app' whatsapp variants:")
print(beh["app"][beh["app"].str.lower().str.contains("what", na=False)].value_counts())
print()
print("Canonical 'app_name' whatsapp variants:")
print(beh["app_name"][beh["app_name"].str.lower().str.contains("what", na=False)].value_counts())

### 2.3 Create event_outcome_simple

Map raw `resolution` values to a simplified canonical column.

The two undocumented values (`dismissedByStructuredIntervention`, `dismissedWithHealthyAlternative`)
are mapped to `dismissed_other` pending codebook confirmation.
They represent 812 rows (0.03%) and are flagged separately for traceability.

In [ ]:
outcome_map = {
    "openedApp":                         "opened",
    "dismissedAppOpening":               "dismissed",
    "closedApp":                         "closed",
    "dismissedByStructuredIntervention": "dismissed_other",   # undocumented
    "dismissedWithHealthyAlternative":   "dismissed_other",   # undocumented
}

beh["event_outcome_raw"]    = beh["resolution"]
beh["event_outcome_simple"] = beh["resolution"].map(outcome_map)

print("event_outcome_simple value counts:")
vc  = beh["event_outcome_simple"].value_counts(dropna=False)
pct = beh["event_outcome_simple"].value_counts(dropna=False, normalize=True) * 100
for val in vc.index:
    print(f"  {str(val):<20s} {vc[val]:>10,}  ({pct[val]:.1f}%)")
print()
print(f"Unmapped (NaN outcome_simple): {beh['event_outcome_simple'].isna().sum()}")

### 2.4 Classify intervention_event_class

All 1,171,929 `closedApp` events have NaN `interventionType`.
This column classifies rows by structural event type:
- `intervention_event` — rows with non-NaN interventionType
- `app_close_event`    — closedApp rows with NaN interventionType
- `no_intervention`    — other rows missing intervention fields

In [ ]:
def classify_event(row):
    if pd.notna(row["interventionType"]):
        return "intervention_event"
    elif row["resolution"] == "closedApp":
        return "app_close_event"
    else:
        return "no_intervention"

beh["intervention_event_class"] = beh.apply(classify_event, axis=1)

print("intervention_event_class distribution:")
vc  = beh["intervention_event_class"].value_counts(dropna=False)
pct = beh["intervention_event_class"].value_counts(dropna=False, normalize=True) * 100
for val in vc.index:
    print(f"  {val:<25s} {vc[val]:>10,}  ({pct[val]:.1f}%)")

### 2.5 Create days_since_first_event

`secondsSinceStudySetup` is inverted (all <= 0) and cannot be used as documented.
Derive a clean forward-counting time variable from `timestamp_utc` directly.

In [ ]:
first_event = beh.groupby("participantUuid")["timestamp_utc"].transform("min")
beh["days_since_first_event"] = (beh["timestamp_utc"] - first_event).dt.total_seconds() / 86400

print("days_since_first_event stats:")
print(beh["days_since_first_event"].describe().round(2))
print()
print("Max days per participant (top 5):")
print(beh.groupby("participantUuid")["days_since_first_event"].max().nlargest(5).round(1))

### 2.6 Add participant_id_canonical

In [ ]:
beh["participant_id_canonical"] = beh["participantUuid"]
print(f"Canonical ID added. Unique participants: {beh['participant_id_canonical'].nunique():,}")

### 2.7 Behavioral cleaning summary

In [ ]:
print("=" * 55)
print("BEHAVIORAL CLEANING SUMMARY")
print("=" * 55)
print(f"  Raw rows:            {len(beh_raw):,}")
print(f"  Cleaned rows:        {len(beh):,}  (no rows dropped)")
print(f"  Columns raw:         {beh_raw.shape[1]}")
print(f"  Columns cleaned:     {beh.shape[1]}")
new_cols = [c for c in beh.columns if c not in beh_raw.columns]
print(f"  New columns:         {new_cols}")
print(f"  NaN in core cols:    {beh[['app_name','resolution','timestamp_utc','participant_id_canonical']].isna().sum().sum()}")

---
## 3. Survey Cleaning
### 3.1 Strip trailing whitespace

In [ ]:
survey = survey_raw.copy()
text_cols = survey.select_dtypes(include="object").columns.tolist()
for col in text_cols:
    survey[col] = survey[col].str.strip()
print(f"Stripped whitespace from {len(text_cols)} text columns.")

### 3.2 Deduplicate — keep first row per participant

5 duplicate uuid rows confirmed. Rule: keep first occurrence.

In [ ]:
n_before = len(survey)
survey = survey.drop_duplicates(subset="uuid", keep="first").reset_index(drop=True)
print(f"Rows before dedup: {n_before}  |  After: {len(survey)}  |  Removed: {n_before - len(survey)}")

### 3.3 Strip survey UUID prefix and add participant_id_canonical

Survey `uuid` has prefix `donatus-master-thesis-one-sec-early-2023-`.
Strip it to create a canonical ID matching behavioral `participantUuid`.

In [ ]:
PREFIX = "donatus-master-thesis-one-sec-early-2023-"
survey["uuid_stripped"]          = survey["uuid"].str.replace(PREFIX, "", regex=False).str.upper()
survey["participant_id_canonical"] = survey["uuid_stripped"]

beh_ids = set(beh["participant_id_canonical"].str.upper())
matched   = survey["participant_id_canonical"].isin(beh_ids).sum()
unmatched = (~survey["participant_id_canonical"].isin(beh_ids)).sum()
print(f"Survey matched to behavioral:    {matched}")
print(f"Survey with no behavioral match: {unmatched}")

### 3.4 Recode gender

Codebook claims numeric codes but data contains text strings.
Apply text-based canonical recoding.

In [ ]:
print("Raw gender values:")
print(survey["gender"].value_counts(dropna=False))
print()

# F043: the previous keys "Non-binary" / "Prefer not to say" never occur in the raw file,
# so 13 non-null responses (11 "Diverse", 2 "Do not want to say") were silently mapped to
# NaN. Keys verified against post_survey_data_set_4.csv 2026-08-08:
# Male 159, Female 82, Diverse 11, Other: 7, Do not want to say 2, NaN 43.
gender_map = {
    "Male":               "male",
    "Female":             "female",
    "Diverse":            "non_binary",
    "Other:":             "other",
    "Do not want to say": "prefer_not_to_say",
}
survey["gender_clean"] = survey["gender"].map(gender_map)

print("gender_clean after recoding:")
print(survey["gender_clean"].value_counts(dropna=False))

# F043: fail loudly instead of warning — a silent NaN here is indistinguishable from a
# genuine non-response and is how the 13 lost values went unnoticed.
unmapped = survey["gender"].notna() & survey["gender_clean"].isna()
assert unmapped.sum() == 0, (
    f"gender_map missed {unmapped.sum()} non-null values: "
    f"{survey[unmapped]['gender'].value_counts().to_dict()}"
)

### 3.5 Survey cleaning summary

In [ ]:
print("=" * 55)
print("SURVEY CLEANING SUMMARY")
print("=" * 55)
print(f"  Raw rows:         {len(survey_raw)}")
print(f"  After dedup:      {len(survey)}")
new_cols_s = [c for c in survey.columns if c not in survey_raw.columns]
print(f"  New columns:      {new_cols_s}")
print(f"  Matched to beh:   {survey['participant_id_canonical'].isin(beh_ids).sum()}")
print(f"  Survey only:      {(~survey['participant_id_canonical'].isin(beh_ids)).sum()}")

---
## 4. Participant Linkage Table

In [ ]:
beh_ids_set    = set(beh["participant_id_canonical"])
survey_ids_set = set(survey["participant_id_canonical"])
all_ids        = beh_ids_set | survey_ids_set

linkage = pd.DataFrame({"participant_id_canonical": sorted(all_ids)})
linkage["in_behavioral"] = linkage["participant_id_canonical"].isin(beh_ids_set)
linkage["in_survey"]     = linkage["participant_id_canonical"].isin(survey_ids_set)
linkage["n_files"]       = linkage["in_behavioral"].astype(int) + linkage["in_survey"].astype(int)

print(f"Linkage table: {len(linkage)} unique participants")
print()
print(f"  Behavioral + survey:  {(linkage['n_files'] == 2).sum():,}")
print(f"  Behavioral only:      {(linkage['in_behavioral'] & ~linkage['in_survey']).sum():,}")
print(f"  Survey only:          {(~linkage['in_behavioral'] & linkage['in_survey']).sum():,}")

---
## 5. Validation Checks

In [ ]:
print("=" * 55)
print("VALIDATION — BEHAVIORAL")
print("=" * 55)
assert beh["participant_id_canonical"].notna().all(), "FAIL: NaN participant_id_canonical"
assert beh["timestamp_utc"].notna().all(), "FAIL: NaN timestamp_utc"
assert beh["app_name"].notna().all(), "FAIL: NaN app_name"
assert beh["event_outcome_simple"].notna().all(), "FAIL: unmapped resolution values"
assert beh["intervention_event_class"].notna().all(), "FAIL: unclassified events"
assert beh["days_since_first_event"].ge(0).all(), "FAIL: negative days_since_first_event"
assert beh.duplicated().sum() == 0, "FAIL: duplicate rows"
print("All behavioral assertions passed.")

In [ ]:
print("=" * 55)
print("VALIDATION — SURVEY")
print("=" * 55)
assert survey.duplicated(subset="participant_id_canonical").sum() == 0, "FAIL: duplicate canonical IDs"
assert survey["participant_id_canonical"].notna().all(), "FAIL: NaN participant_id_canonical"
print("All survey assertions passed.")

In [ ]:
print("=" * 55)
print("VALIDATION — LINKAGE")
print("=" * 55)
assert len(linkage) == len(all_ids), "FAIL: linkage row count mismatch"
assert linkage["participant_id_canonical"].nunique() == len(linkage), "FAIL: duplicate IDs in linkage"
assert (linkage["n_files"] >= 1).all(), "FAIL: participant with 0 files"
print("All linkage assertions passed.")

---
## 6. Save Cleaned Outputs

In [ ]:
BEH_OUT     = CLEANED / "ds4_behavioral_clean.csv"
SURVEY_OUT  = CLEANED / "ds4_survey_clean.csv"
LINKAGE_OUT = CLEANED / "ds4_participant_linkage.csv"

beh.to_csv(BEH_OUT, index=False)
survey.to_csv(SURVEY_OUT, index=False)
linkage.to_csv(LINKAGE_OUT, index=False)

print(f"Saved: {BEH_OUT.name}     ({BEH_OUT.stat().st_size/1024/1024:.1f} MB, {len(beh):,} rows)")
print(f"Saved: {SURVEY_OUT.name}  ({SURVEY_OUT.stat().st_size:.0f} bytes, {len(survey)} rows)")
print(f"Saved: {LINKAGE_OUT.name} ({LINKAGE_OUT.stat().st_size:.0f} bytes, {len(linkage)} rows)")

---
## 7. Final Cleaning Summary

In [ ]:
print("=" * 65)
print("DS4 CLEANING COMPLETE")
print("=" * 65)
print()
print("BEHAVIORAL")
print(f"  Raw rows:              {len(beh_raw):,}")
print(f"  Cleaned rows:          {len(beh):,}  (no rows dropped)")
print(f"  New columns added:     7")
print(f"  Unique participants:   {beh['participant_id_canonical'].nunique():,}")
print()
print("SURVEY")
print(f"  Raw rows:              {len(survey_raw)}")
print(f"  After dedup:           {len(survey)}")
print(f"  New columns added:     3  (uuid_stripped, participant_id_canonical, gender_clean)")
print()
print("LINKAGE")
print(f"  Total unique:          {len(linkage):,}")
print(f"  Behavioral + survey:   {(linkage['n_files']==2).sum()}")
print(f"  Behavioral only:       {(linkage['in_behavioral'] & ~linkage['in_survey']).sum():,}")
print(f"  Survey only:           {(~linkage['in_behavioral'] & linkage['in_survey']).sum()}")
print()
print("OUTPUT FILES")
print("  outputs/cleaned/ds4_behavioral_clean.csv")
print("  outputs/cleaned/ds4_survey_clean.csv")
print("  outputs/cleaned/ds4_participant_linkage.csv")
print()
print("IMPORTANT CAVEATS")
print("  1. dismissedByStructuredIntervention + dismissedWithHealthyAlternative")
print("     mapped to 'dismissed_other' — confirm before treating as dismissed")
print("  2. intervention_event_class separates closedApp from active intervention events")
print("  3. days_since_first_event replaces unusable secondsSinceStudySetup")
print("  4. 756 behavioral-only participants have no psychosocial variables")